##LOAD SILVER LAYER

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ====================================================================
# Create Silver Schema
# ====================================================================

spark.sql("""
CREATE SCHEMA IF NOT EXISTS olist_lakehouse.silver
""")

# ====================================================================
# Reusable Text Normalization Function
# ====================================================================

def normalize_text(column):

    return lower(
        trim(
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        translate(
                            column,
                            'áàãâéêíóôõúç',
                            'aaaaeeiooouc'
                        ),
                        "'",
                        ""
                    ),
                    '"',
                    ""
                ),
                r"\*|\.\.",
                ""
            )
        )
    )

# ####################################################################
# 1. LOAD CUSTOMERS DATASET
# ####################################################################

print("Loading: silver.olist_customers_dataset")

customers_df = (
    spark.table("olist_lakehouse.bronze.olist_customers_dataset")
    .select(
        regexp_replace(trim(col("customer_id")), '"', '')
            .alias("customer_id"),

        regexp_replace(trim(col("customer_unique_id")), '"', '')
            .alias("customer_unique_id"),

        regexp_replace(trim(col("customer_zip_code_prefix")), '"', '')
            .alias("customer_zip_code_prefix"),

        normalize_text(col("customer_city"))
            .alias("customer_city"),

        trim(col("customer_state"))
            .alias("customer_state"),

        current_timestamp()
            .alias("dwh_create_date")
    )
)

(
    customers_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_lakehouse.silver.olist_customers_dataset")
)

print(f"Rows Loaded: {customers_df.count()}")


# ####################################################################
# 2. LOAD GEOLOCATION DATASET
# ####################################################################

print("Loading: silver.olist_geolocation_dataset")

window_spec = Window.partitionBy("geolocation_zip_code_prefix")

geo_df = (
    spark.table("olist_lakehouse.bronze.olist_geolocation_dataset")
    .withColumn(
        "geolocation_lat",
        avg("geolocation_lat").over(window_spec)
    )
    .withColumn(
        "geolocation_lng",
        avg("geolocation_lng").over(window_spec)
    )
    .withColumn(
        "row_num",
        row_number().over(
            Window.partitionBy("geolocation_zip_code_prefix")
            .orderBy("geolocation_zip_code_prefix")
        )
    )
    .filter(col("row_num") == 1)
    .select(

        normalize_text(col("geolocation_zip_code_prefix"))
            .alias("geolocation_zip_code_prefix"),

        col("geolocation_lat"),
        col("geolocation_lng"),

        normalize_text(col("geolocation_city"))
            .alias("geolocation_city"),

        trim(col("geolocation_state"))
            .alias("geolocation_state"),

        current_timestamp()
            .alias("dwh_create_date")
    )
)

(
    geo_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_lakehouse.silver.olist_geolocation_dataset")
)

print(f"Rows Loaded: {geo_df.count()}")


# ####################################################################
# 3. LOAD ORDER ITEMS DATASET
# ####################################################################

print("Loading: silver.olist_order_items_dataset")

order_items_df = (
    spark.table("olist_lakehouse.bronze.olist_order_items_dataset")
    .select(

        regexp_replace(trim(col("order_id")), '"', '')
            .alias("order_id"),

        col("order_item_id"),

        regexp_replace(trim(col("product_id")), '"', '')
            .alias("product_id"),

        regexp_replace(trim(col("seller_id")), '"', '')
            .alias("seller_id"),

        col("shipping_limit_date"),
        col("price"),
        col("freight_value"),

        current_timestamp()
            .alias("dwh_create_date")
    )
)

(
    order_items_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_lakehouse.silver.olist_order_items_dataset")
)

print(f"Rows Loaded: {order_items_df.count()}")


# ####################################################################
# 4. LOAD ORDER PAYMENTS DATASET
# ####################################################################

print("Loading: silver.olist_order_payments_dataset")

payments_df = (
    spark.table("olist_lakehouse.bronze.olist_order_payments_dataset")
    .select(

        regexp_replace(trim(col("order_id")), '"', '')
            .alias("order_id"),

        col("payment_sequential").cast("int")
            .alias("payment_sequential"),

        trim(col("payment_type"))
            .alias("payment_type"),

        col("payment_installments"),
        col("payment_value"),

        current_timestamp()
            .alias("dwh_create_date")
    )
)

(
    payments_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_lakehouse.silver.olist_order_payments_dataset")
)

print(f"Rows Loaded: {payments_df.count()}")


# ####################################################################
# 5. LOAD ORDER REVIEWS DATASET
# ####################################################################

print("Loading: silver.olist_order_reviews_dataset")

reviews_df = (
    spark.table("olist_lakehouse.bronze.olist_order_reviews_dataset")
    .select(

        regexp_replace(trim(col("review_id")), '"', '')
            .alias("review_id"),

        regexp_replace(trim(col("order_id")), '"', '')
            .alias("order_id"),

        col("review_score"),

        trim(
            coalesce(col("review_comment_title"), lit("n/a"))
        ).alias("review_comment_title"),

        trim(
            coalesce(col("review_comment_message"), lit("n/a"))
        ).alias("review_comment_message"),

        col("review_creation_date"),
        col("review_answer_timestamp"),

        current_timestamp()
            .alias("dwh_create_date")
    )
)

(
    reviews_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_lakehouse.silver.olist_order_reviews_dataset")
)

print(f"Rows Loaded: {reviews_df.count()}")


# ####################################################################
# 6. LOAD ORDERS DATASET
# ####################################################################

print("Loading: silver.olist_orders_dataset")

orders_df = (
    spark.table("olist_lakehouse.bronze.olist_orders_dataset")
    .select(

        regexp_replace(trim(col("order_id")), '"', '')
            .alias("order_id"),

        regexp_replace(trim(col("customer_id")), '"', '')
            .alias("customer_id"),

        trim(col("order_status"))
            .alias("order_status"),

        col("order_purchase_timestamp"),

        when(
            (col("order_approved_at") < col("order_purchase_timestamp")) &
            (
                abs(
                    unix_timestamp("order_approved_at")
                    - unix_timestamp("order_purchase_timestamp")
                ) <= 3600
            ),
            col("order_purchase_timestamp")
        ).otherwise(col("order_approved_at"))
            .alias("order_approved_at"),

        when(
            (col("order_delivered_carrier_date") < col("order_approved_at")) &
            (
                abs(
                    unix_timestamp("order_delivered_carrier_date")
                    - unix_timestamp("order_approved_at")
                ) <= 3600
            ),
            col("order_approved_at")
        ).otherwise(col("order_delivered_carrier_date"))
            .alias("order_delivered_carrier_date"),

        when(
            (col("order_delivered_customer_date") < col("order_delivered_carrier_date")) &
            (
                abs(
                    unix_timestamp("order_delivered_customer_date")
                    - unix_timestamp("order_delivered_carrier_date")
                ) <= 3600
            ),
            col("order_delivered_carrier_date")
        ).otherwise(col("order_delivered_customer_date"))
            .alias("order_delivered_customer_date"),

        when(
            col("order_estimated_delivery_date") < col("order_purchase_timestamp"),
            col("order_purchase_timestamp")
        ).otherwise(col("order_estimated_delivery_date"))
            .alias("order_estimated_delivery_date"),

        when(
            col("order_approved_at") < col("order_purchase_timestamp"),
            1
        ).otherwise(0)
            .alias("dq_invalid_approval_timestamp_flag"),

        when(
            col("order_delivered_carrier_date") < col("order_approved_at"),
            1
        ).otherwise(0)
            .alias("dq_invalid_carrier_timestamp_flag"),

        when(
            col("order_delivered_customer_date") < col("order_delivered_carrier_date"),
            1
        ).otherwise(0)
            .alias("dq_invalid_customer_delivery_timestamp_flag"),

        when(
            col("order_estimated_delivery_date") < col("order_purchase_timestamp"),
            1
        ).otherwise(0)
            .alias("dq_invalid_estimated_delivery_timestamp_flag"),

        current_timestamp()
            .alias("dwh_create_date")
    )
)

(
    orders_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_lakehouse.silver.olist_orders_dataset")
)

print(f"Rows Loaded: {orders_df.count()}")


# ####################################################################
# 7. LOAD PRODUCTS DATASET
# ####################################################################

print("Loading: silver.olist_products_dataset")

products_df = (
    spark.table("olist_lakehouse.bronze.olist_products_dataset")
    .select(

        regexp_replace(trim(col("product_id")), '"', '')
            .alias("product_id"),

        when(
            col("product_category_name").isNull(),
            "n/a"
        ).otherwise(trim(col("product_category_name")))
            .alias("product_category_name"),

        coalesce(col("product_name_lenght"), lit(0))
            .alias("product_name_lenght"),

        coalesce(col("product_description_lenght"), lit(0))
            .alias("product_description_lenght"),

        coalesce(col("product_photos_qty"), lit(0))
            .alias("product_photos_qty"),

        coalesce(col("product_weight_g"), lit(0))
            .alias("product_weight_g"),

        coalesce(col("product_length_cm"), lit(0))
            .alias("product_length_cm"),

        coalesce(col("product_height_cm"), lit(0))
            .alias("product_height_cm"),

        coalesce(col("product_width_cm"), lit(0))
            .alias("product_width_cm"),

        current_timestamp()
            .alias("dwh_create_date")
    )
)

(
    products_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_lakehouse.silver.olist_products_dataset")
)

print(f"Rows Loaded: {products_df.count()}")


# ####################################################################
# 8. LOAD SELLERS DATASET
# ####################################################################

print("Loading: silver.olist_sellers_dataset")

sellers_df = (
    spark.table("olist_lakehouse.bronze.olist_sellers_dataset")
    .select(

        regexp_replace(trim(col("seller_id")), '"', '')
            .alias("seller_id"),

        regexp_replace(trim(col("seller_zip_code_prefix")), '"', '')
            .alias("seller_zip_code_prefix"),

        normalize_text(col("seller_city"))
            .alias("seller_city"),

        trim(col("seller_state"))
            .alias("seller_state"),

        current_timestamp()
            .alias("dwh_create_date")
    )
)

(
    sellers_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_lakehouse.silver.olist_sellers_dataset")
)

print(f"Rows Loaded: {sellers_df.count()}")


# ####################################################################
# 9. LOAD PRODUCT CATEGORY TRANSLATION DATASET
# ####################################################################

print("Loading: silver.product_category_name_translation")

translation_df = (
    spark.table("olist_lakehouse.bronze.product_category_name_translation")
    .select(

        trim(col("product_category_name"))
            .alias("product_category_name"),

        trim(col("product_category_name_english"))
            .alias("product_category_name_english"),

        current_timestamp()
            .alias("dwh_create_date")
    )
)

(
    translation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "olist_lakehouse.silver.product_category_name_translation"
    )
)

print(f"Rows Loaded: {translation_df.count()}")


# ####################################################################
# COMPLETE
# ####################################################################

print("==================================================")
print("ALL SILVER TABLES LOADED SUCCESSFULLY")
print("==================================================")